# 02 - Healthcare Data Engineering

This notebook performs the extraction, preprocessing, and aggregation of healthcare information used to enrich the patient-level dataset generated in Notebook 01. It includes laboratory tests, diagnoses, pharmacological treatments, primary care visits, hospitalizations, and emergency department visits.

**Objectives**:

- Extract healthcare records within each patient's observation window.
- Aggregate laboratory tests into clinically meaningful groups.
- Group diagnoses according to ICD-10 categories.
- Aggregate pharmacological treatments.
- Summarize healthcare utilization variables.
- Merge all healthcare information into a unified patient-level dataset.
- Export the final dataset used for model development.

## Setup

In [1]:
# import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import unicodedata
from utils import extract_data, extract_data_parallel, build_patient_window_sql, create_chunks

In [2]:
# dblinks from Tarragona i Reus
dblinks = ['@P6211_exp']

In [3]:
# Load data
cip_extraction = pd.read_csv('data/cip_relations.csv')
df_usuaris = pd.read_csv('data/df_usuaris.csv')

In [4]:
cip_extraction['end_window'] = pd.to_datetime(cip_extraction['end_window'], errors='coerce')
cip_extraction['start_window'] = pd.to_datetime(cip_extraction['start_window'], errors='coerce')

## 1. Laboratory Extraction

### 1.1 SQL Query for Laboratory Tests

In [68]:
# Create chunks of patients as temporal SQL tables
chunk_size = 800
user_chunks_lab = create_chunks(cip_extraction, chunk_size, 'cip_anterior_14')

In [70]:
q_lab = """

 -- Insert the generated SQL as a Common Table Expression (CTE)
WITH patient_window AS (
    {usuaris}
)

SELECT 
    lab.cr_cip AS cip_anterior_14,
    lab.cr_data_reg AS data_resultat_lab,
    lab.cr_desc_prova_ics AS desc_prova_ics,
    lab.cr_res_lab AS resultat_lab

FROM labtb101{dbl} lab

-- Join with the generated table
JOIN patient_window p
    ON lab.cr_cip = p.cip

-- Only obtain the necessary lab tests
WHERE lab.cr_data_reg BETWEEN p.start_window AND p.end_window
"""

In [71]:
df_lab = extract_data_parallel(
    q_lab,
    dblinks,
    user_chunks=user_chunks_lab,
    max_workers=6
)

print(f'Number of tests: {df_lab.shape[0]}')

Number of tests: 4544741


In [ ]:
# Drop dbl column and duplicates
df_lab = df_lab.drop(columns='dbl')
df_lab = (df_lab.drop_duplicates(keep='first')).reset_index(drop=True)
print(df_lab.shape[0])

3361720


Merge the lab tests with *cip_extraction* to get the *id_cip_actual* of each test:

In [74]:
df_lab_2 = pd.merge(
    df_lab, 
    cip_extraction[['id_cip_actual', 'cip_anterior_14', 'start_window', 'end_window']],
    on='cip_anterior_14',
    how='left'
    )

df_lab_2.insert(0, 'id_cip_actual', df_lab_2.pop('id_cip_actual'))

# Calculate the data taking start_window as day 1
df_lab_2['data'] = (
    pd.to_datetime(df_lab_2['data_resultat_lab']) - pd.to_datetime(df_lab_2['start_window'])
    ).dt.days
df_lab_2.drop(columns=['cip_anterior_14', 'data_resultat_lab', 'start_window', 'end_window'], inplace=True)
df_lab_2.insert(1, 'data', df_lab_2.pop('data'))

print(f'Number of lab tests: {df_lab_2.shape[0]}')
df_lab_2.head()

Number of lab tests: 3361720


,id_cip_actual,data,desc_prova_ics,resultat_lab
0,2,111,TEMPS DE PROTROMBINA (SEGONS); TEMPS,"11,9"
1,2,111,TEMPS DE PROTROMBINA (RATI); TEMPS,"1,03"
2,2,111,INR; TEMPS REL,"1,03"
3,2,111,TTPA (SEGONS); TEMPS,"31,1"
4,2,111,TTPA (RATI); TEMPS,"0,99"


### 1.2 Laboratory Tests Groupings

Define a list of the lab tests done for actue illness and another list of lab tests realized on chronic patients:

In [75]:
proves_aguts = ['PRO-BNP-SÈRUM',
                'PROTEÏNA C REACTIVA (PCR)-SÈRUM',
                'ERITROSEDIMENTACIÓ (VSG)-SANG',
                'DÍMER D DE LA FIBRINA (IMMUNOTURBIDIMETRIA)-PLASMA']

proves_cronics = ['PERFIL-HEMOGRAMA-SANG',
                  'GLUCOSA-SÈRUM',
                  'CERATININI-SÈRUM',
                  'UREA-SÈRUM',
                  'PERFIL-ESTIMACIÓ FILTRAT GLOMERULAR (SEGONS CKD-EPI)-SÈRUM',
                  'PERFIL-IONS (SODI, POTASSI I CLORUR)-SÈRUM',
                  'ASPARTAT AMINOTRANSFERASA-SÈRUM',
                  'ALANINA AMINOTRANSFERASA-SÈRUM',
                  'GAMMA-GLUTANIL TRANSFERASA-SÈRUM',
                  'BILIRUBINA-SÈRUM',
                  'FERRO-SÈRUM',
                  'FERRITINA-SÈRUM',
                  'FERRITINA',
                  'ÀCID FÒLIC-SÈRUM',
                  'COBALAMINES (VITAMINA B12)-SÈRUM',
                  'PROTEÏNA-SÈRUM',
                  'ALBÚMINA-SÈRUM',
                  'COLESTEROL-SÈRUM',
                  'TIROTROPINA-SÈRUM',
                  'FOSFATASA ALCALINA-SÈRUM']

In [76]:
# Filter acute lab tests
df_lab_aguts = df_lab_2[df_lab_2['desc_prova_ics'].isin(proves_aguts)]
print(f'Number of acute illness tests: {df_lab_aguts.shape[0]}')

# Filter chronic lab tests:
df_lab_cronic = df_lab_2[df_lab_2['desc_prova_ics'].isin(proves_cronics)]
print(f'Number of cronic patients tests: {df_lab_cronic.shape[0]}')

# Merge both extractions
df_lab_tests = pd.concat([df_lab_cronic, df_lab_aguts]).reset_index(drop=True)
print(f'Total number of lab tests: {df_lab_tests.shape[0]}')

# Check NaN values in results
print(f'NaN values in lab results: {df_lab_tests.resultat_lab.isna().sum()}')

df_lab_tests.head()

Number of acute illness tests: 6051
Number of cronic patients tests: 26974
Total number of lab tests: 33025
NaN values in lab results: 0


,id_cip_actual,data,desc_prova_ics,resultat_lab
0,21,33,GLUCOSA-SÈRUM,123
1,21,33,UREA-SÈRUM,64.1
2,21,296,GLUCOSA-SÈRUM,60
3,21,296,UREA-SÈRUM,89
4,73,113,ALANINA AMINOTRANSFERASA-SÈRUM,49


#### 1.2.1 Handle non-numeric values in lab results

There are non-numeric values on lab results, so they need to be managed:
- Rows where lab results do not have any numeric value ar discarded.
- Handle lab results writed as: >8, <8, "8", 1,8, lt;8
    1. Case >8: consider a value slightly above the thershold.
    2. Case <8: consider a value slightly below the thershold.
    3. Case "8": remove ""
    4. Case 1,8: change the , for a .
    5. Case lt;8: it is the html character for <, so we change lt; for <
    5. Case gt;8: it is the html character for >, so we change for >

In [77]:
# Keep only the rows where resultat_lab contains at least one digit
df_lab_tests = df_lab_tests[
    df_lab_tests['resultat_lab'].str.contains(r'\d', na=False)
].reset_index(drop=True)

print(f'Number of lab tests: {df_lab_tests.shape[0]}')

Number of lab tests: 32570


In [78]:
# column resultat_lab as string
s = df_lab_tests['resultat_lab'].astype(str)

# Replace string characters to get numeric values
df_lab_tests['resultat_lab'] = pd.to_numeric(
    s.str.replace('lt;', '<', regex=False)
    .str.replace('gt;', '>', regex=False)
    .str.replace('"', '', regex=False)
    .str.replace('[<>]', '', regex=True)
    .str.replace(',', '.', regex=False),
    errors='coerce'
)

# Small proportional adjustment to perserve the meaning of the inquealities depending on the value of each row
df_lab_tests.loc[s.str.startswith('lt;|<'), 'resultat_lab'] *= 0.99
df_lab_tests.loc[s.str.startswith('gt;|>'), 'resultat_lab'] *= 1.01

# Drop the remaining rows where resultats_lab are NaN
df_lab_tests.dropna(subset=['resultat_lab'], inplace=True)
print(f'Number of lab tests: {df_lab_tests.shape[0]}')

Number of lab tests: 32569


### 1.3 Statisctical measures for lab results

The trend across 12-month data of each lab test for each patient is saved using:
- Mean
- Median
- Standard Deviation
- Slope

In [79]:
# Function to compute slope using linear regression

def compute_slope(x,y):
    if len(x) < 2 or x.nunique() < 2:
        return np.nan       # Can't compute the slope with only one point of data
    else:
        return np.polyfit(x, y, 1)[0]       # Linear regressiob degree=1

To compute the slope the date of the results must be a numeric column:
- The minimum date is considered as the initial point (0).
- The days between the minimum date and the actual date in the row are calculated.
- This will represent how *resultats_lab* change over time.

In [80]:
# Groupby patient and test, calculate the statictics for each group
df_lab_measures = (
    df_lab_tests.groupby(['id_cip_actual', 'desc_prova_ics'])
    .apply(lambda g: pd.Series({
        'mean': np.round(g['resultat_lab'].mean(), 3),
        'last': np.round(g.sort_values('data')['resultat_lab'].iloc[-1], 3),
        'slope': np.round(compute_slope(g['data'], g['resultat_lab']), 3)
    }))
    .reset_index()
    .fillna(0)                                                          # non-values asvalue 0 for that statistic measure
)
print(f'Number of lab tests: {df_lab_measures.shape[0]}')
df_lab_measures.head(15)

Number of lab tests: 15277


,id_cip_actual,desc_prova_ics,mean,last,slope
0,21,GLUCOSA-SÈRUM,91.500,60.00,-0.240
1,21,PROTEÏNA C REACTIVA (PCR)-SÈRUM,3.445,2.49,-0.007
2,21,UREA-SÈRUM,76.550,89.00,0.095
3,73,ALANINA AMINOTRANSFERASA-SÈRUM,22.333,11.00,-0.168
4,73,ALBÚMINA-SÈRUM,3.600,3.60,0.000
5,73,ASPARTAT AMINOTRANSFERASA-SÈRUM,40.333,27.00,-0.157
6,73,BILIRUBINA-SÈRUM,0.767,0.70,-0.000
7,73,GLUCOSA-SÈRUM,117.667,148.00,0.233
8,73,PROTEÏNA C REACTIVA (PCR)-SÈRUM,5.267,6.60,0.002
9,73,UREA-SÈRUM,72.300,140.80,0.196


### 1.4 Pivoting

>Transform data from long to patient-level format:

- Laboratory tests are structured at the individual record level, where each row represents a single laboratory measurement performed for a patient.
- To enable predictive modeling, laboratory measurements are aggregated so that each patient is represented by a single row.

>Aggregate laboratory measurements:

- Only clinically relevant laboratory tests are retained.
- Laboratory measurements are aggregated at the patient level by computing summary statistics and temporal trend features for each laboratory parameter.
- The value of the last available measurement is also preserved to retain recent clinical information.
- Missing values are filled as 0.

In [81]:
# From wide to long format
df_lab_final = (
    df_lab_measures.pivot(
        index='id_cip_actual', 
        columns='desc_prova_ics', 
        values=['mean', 'slope', 'last'])
    .fillna(0)
)

df_lab_final.columns = [
    f"{stat}_{test}" for stat, test in df_lab_final.columns
]

df_lab_final = df_lab_final.reset_index()
df_lab_final.columns = df_lab_final.columns.str.lower()

In [82]:
print(f'Number of measure of laboratory tests: {df_lab_final.shape[1]}')
print(f'Number patients with laboratory tests: {df_lab_final.shape[0]}')
df_lab_final.to_csv('data/df_lab.csv', index=False)
df_lab_final.head()

Number of measure of laboratory tests: 55
Number patients with laboratory tests: 3338


,id_cip_actual,mean_alanina aminotransferasa-sèrum,mean_albúmina-sèrum,mean_aspartat aminotransferasa-sèrum,mean_bilirubina-sèrum,mean_cobalamines (vitamina b12)-sèrum,mean_colesterol-sèrum,mean_dímer d de la fibrina (immunoturbidimetria)-plasma,mean_eritrosedimentació (vsg)-sang,mean_ferritina-sèrum,...,last_ferritina-sèrum,last_ferro-sèrum,last_fosfatasa alcalina-sèrum,last_glucosa-sèrum,last_pro-bnp-sèrum,last_proteïna c reactiva (pcr)-sèrum,last_proteïna-sèrum,last_tirotropina-sèrum,last_urea-sèrum,last_àcid fòlic-sèrum
0,21,0.000,0.0,0.000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,60.0,0.0,2.49,0.0,0.00,89.0,0.0
1,73,22.333,3.6,40.333,0.767,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,148.0,0.0,6.60,0.0,0.00,140.8,0.0
2,91,0.000,0.0,0.000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.20,0.0,0.00,52.3,0.0
3,153,24.000,0.0,32.500,1.197,293.0,0.0,0.0,0.0,214.0,...,214.0,9.0,126.0,160.0,0.0,0.00,0.0,0.35,0.0,1.8
4,226,0.000,0.0,0.000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.48,0.0,0.00,32.7,0.0


## 2. Diagnoses Extraction

### 2.1 SQL Query for Diagnoses

In [5]:
# Create chunks of patients as temporal SQL tables
chunk_size = 800
user_chunks_dx = create_chunks(cip_extraction, chunk_size, 'cip_anterior')

In [6]:
q_dx = """

 -- Insert the generated SQL as a Common Table Expression (CTE)
WITH patient_window AS (
    {usuaris}
)

SELECT
    dx.pr_cod_u AS cip_anterior,
    dx.pr_cod_ps AS codi_dx,
    dx.pr_cod_ps_cim10 AS codi_CIM10,
    TO_CHAR(dx.pr_dde, 'YYYY-MM-DD') AS data_dx,
    TO_CHAR(dx.pr_dba, 'YYYY-MM-DD') AS data_baixa_dx

FROM prstb015{dbl} dx

-- Join with the generated table
JOIN patient_window p
    ON dx.pr_cod_u = p.cip

-- Only obtain the diagnoses in the temporal window of each patient
WHERE dx.pr_dde <= p.end_window
  AND (
        dx.pr_dba IS NULL 
        OR pr_dba >= p.start_window
        )
"""

In [7]:
df_dx = extract_data_parallel(
    q_dx,
    dblinks,
    user_chunks=user_chunks_dx,
    max_workers=6
)

print(f'Number of diagnoses: {df_dx.shape[0]}')

Number of diagnoses: 566139


In [8]:
# Remove non-required symbols of the CIM10 diagnose codes
df_dx['codi_cim10'] = df_dx['codi_cim10'].str.replace('.','')

Merge diagnoses with *cip_extraction* to get the *id_cip_actual* of each diagnose:

In [12]:
df_dx_2 = pd.merge(
    df_dx, 
    cip_extraction[['id_cip_actual', 'cip_anterior', 'start_window']],
    on='cip_anterior',
    how='left'
    )

df_dx_2.insert(0, 'id_cip_actual', df_dx_2.pop('id_cip_actual'))

# Calculate the data taking start_window as day 1
df_dx_2['data_dx'] = (
    pd.to_datetime(df_dx_2['data_dx']) - pd.to_datetime(df_dx_2['start_window'])
    ).dt.days
df_dx_2['data_baixa_dx'] = (
    pd.to_datetime(df_dx_2['data_baixa_dx']) - pd.to_datetime(df_dx_2['start_window'])
    ).dt.days

df_dx_2.drop(columns=['cip_anterior', 'start_window', 'dbl'], inplace=True)

print(f'Number of diagnoses: {df_dx_2.shape[0]}')
df_dx_2.head()

Number of diagnoses: 566139


,id_cip_actual,codi_dx,codi_cim10,data_dx,data_baixa_dx
0,2,C01-E78.9,E78,-8191,NaN
1,2,C01-T14.8XXA,NaN,133,193.0
2,2,C01-G62.9,NaN,-563,NaN
3,2,C01-H25.9,H259,-3132,NaN
4,2,C01-H40.9,H409,-4197,NaN


### 2.2 Diagnoses Groupings

From the file with diagnoses grouping, only the CIM10 code of each diagnose and the group where it belongs to are selected:

In [13]:
# Load the file that contains the diagnoses agrupations
dx_grups = pd.read_excel('data/dx agrupadors.xlsx')

# Delete rows where any value of the 2 interesting columns is NaN
dx_grups = dx_grups[['grup','ACCOUNTS_Symbol']].dropna()

# Rename the CIM10 diagnose code column for merging
dx_grups = dx_grups.rename(columns={'ACCOUNTS_Symbol': 'codi_cim10'})

# Drop possible duplicates and ensure data type
dx_grups = dx_grups.drop_duplicates()
dx_grups['codi_cim10'] = dx_grups['codi_cim10'].astype(str)
dx_grups = dx_grups.reset_index(drop=True)

print(f'Number of diagnoses agrupations: {dx_grups.shape[0]}')
dx_grups.head()

Number of diagnoses agrupations: 857


,grup,codi_cim10
0,Signes_i_sintomes,A231
1,Signes_i_sintomes,A233
2,Signes_i_sintomes,A232
3,Signes_i_sintomes,A01
4,Signes_i_sintomes,A02


Diagnoses codes follow a hierarchical structure, where more specific codes are extensions of more general ones, so the same clinical category can be represented by codes of different lengths. The grouping table does not always contain all possible specific codes, but rather their more general prefixes. Therefore, an exact match between datasets is not sufficient to assign groups correctly.  
To address this, a prefix-based approach is required, allowing each diagnosis to be linked to the most appropriate group by matching it to the closest valid prefix.  
This ensures that all diagnoses are classified, while preserving the highest level of specificity available.

In [14]:
# Select the possible codes of extracted diagnose data
codes = df_dx_2[['codi_cim10']].dropna().drop_duplicates().reset_index(drop=True)
codes['codi_cim10'] = codes['codi_cim10'].astype(str)

# Prepare result column to assign the groups
codes['grup'] = pd.NA

# Get the unique possible string lengths of CIM10 codes in descending order
lengths = sorted(dx_grups['codi_cim10'].str.len().unique(), reverse=True)

for L in lengths:

    # Only process the unassigned codes
    mask_codes = codes['grup'].isna()
    if not mask_codes.any():
        break

    tmp_codes = codes.loc[mask_codes, ['codi_cim10']].copy()

    # Extract the prefix length of L
    tmp_codes['prefix'] = tmp_codes['codi_cim10'].str[:L]

    # Only compare same-lentgh prefixes
    mask_len = dx_grups['codi_cim10'].str.len() == L

    # Filter mapping table to same lentgh
    tmp_dxg = (
        dx_grups.loc[mask_len, ['codi_cim10', 'grup']]
           .rename(columns={'codi_cim10': 'prefix'})       # rename column for merging
    )

    # Merge the codes with the extracted diagnoses
    matched = tmp_codes.merge(tmp_dxg, on="prefix", how="left")

    # Assign the codes that have matching groups with this prefix length
    codes.loc[mask_codes, 'grup'] = matched['grup'].values

# Assign group to each row in the diagnoses extraction
df_dx_2['grup'] = df_dx_2['codi_cim10'].map(codes.set_index('codi_cim10')['grup'])      # create a dictionary of diagnose code and its group
df_dx_2['grup'] = df_dx_2['grup'].fillna('altres_diags')

>Transform data from long to patient-level format:

- Diagnoses are structured at the individual record level, where each row represents a single diagnosis assigned to a patient.
- To enable analysis, they must be transformed so that each patient is represented by a single row.

>Group diagnoses:

- Since patients may present a large number of different diagnoses, records are aggregated by patient and diagnosis group.
- The number of diagnoses within each group is computed.
- NaN values are replaced with 0, indicating that no diagnoses were recorded for that group.

In [15]:
# From wide to long format
df_dx_grups = (
    df_dx_2.groupby(['id_cip_actual','grup'], as_index=False)               # groups by cip and grup keeping them as regular columns (not become indexes)
    .agg({'codi_cim10': 'nunique'})                                         # number of different diagnose codes per group
    .pivot(index='id_cip_actual', columns='grup', values='codi_cim10')      # from long to wide format (cip as rows, grup values as columns, counts of cim10 codes for grups as values)
    .reset_index()
    .fillna(0)                                                              # non-values as 0 counts for that diagnose group
)

# All columns that are not cip_anterior as integer type
cols = df_dx_grups.columns.difference(['id_cip_actual']) 
df_dx_grups[cols] = df_dx_grups[cols].astype(int)

# Create a column of the total diagnoses for patient
df_dx_grups['Diags_totals'] = df_dx_grups[cols].sum(axis=1)
df_dx_grups.columns = df_dx_grups.columns.str.lower()
df_dx_grups.columns.name = None

In [16]:
df_dx_final = df_dx_grups.copy()
print(f'Number of diagnose groups: {df_dx_final.shape[1]}')
print(f'Number patients with diagnoses: {df_dx_final.shape[0]}')
df_dx_final.to_csv('data/df_diagnoses.csv', index=False)
df_dx_final.head()

Number of diagnose groups: 10
Number patients with diagnoses: 36379


,id_cip_actual,problemes_salut_aguts,problemes_salut_anomalia_congenita,problemes_salut_cronics,problemes_salut_indefinits,problemes_salut_neoplasia_benigna,problemes_salut_neoplasia_maligna,signes_i_sintomes,altres_diags,diags_totals
0,2,0,0,0,0,0,0,0,3,3
1,7,0,0,0,0,0,0,0,7,7
2,8,0,0,0,0,0,0,0,4,4
3,9,0,0,0,0,0,0,0,6,6
4,12,0,0,0,0,0,0,1,8,9


## 3. Treatments Extraction

### 3.1 SQL Query For Treatments

In [17]:
# Create chunks of patients as temporal SQL tables
chunk_size = 800
user_chunks_tract = create_chunks(cip_extraction, chunk_size, 'cip_anterior')

In [18]:
q_tract = """

 -- Insert the generated SQL as a Common Table Expression (CTE)
WITH patient_window AS (
    {usuaris}
)

SELECT
    ppf.ppfmc_pmc_usuari_cip AS cip_anterior,
    ppf.ppfmc_pmc_data_ini AS data_inici,
    ppf.ppfmc_data_fi AS data_fi,
    ppf.ppfmc_atccodi AS atc,
    atc.atc_desc

FROM ppftb016{dbl} ppf

LEFT JOIN cpftb010{dbl} atc
    ON ppf.ppfmc_atccodi = atc.atc_codi

-- Join with the generated table
JOIN patient_window p
    ON ppf.ppfmc_pmc_usuari_cip = p.cip

-- Only obtain the treatments in the temporal window of each patient
WHERE ppf.ppfmc_pmc_data_ini <= p.end_window
    AND (
        ppf.ppfmc_data_fi IS NULL
        OR ppf.ppfmc_data_fi >= p.start_window
        )
"""

In [19]:
df_tract = extract_data_parallel(
    q_tract,
    dblinks,
    user_chunks=user_chunks_tract,
    max_workers=6
)

print(f'Number of treatments: {df_tract.shape[0]}')

Number of treatments: 451998


Merge treatments with *cip_extraction* to get the *id_cip_actual* of each treatment:

In [20]:
df_tract_2 = pd.merge(
    df_tract, 
    cip_extraction[['id_cip_actual', 'cip_anterior', 'start_window']],
    on='cip_anterior',
    how='left'
    )

df_tract_2.insert(0, 'id_cip_actual', df_tract_2.pop('id_cip_actual'))

# Calculate the data taking start_window as day 1
df_tract_2['data_inici'] = (
    pd.to_datetime(df_tract_2['data_inici']) - pd.to_datetime(df_tract_2['start_window'])
    ).dt.days
df_tract_2['data_fi'] = (
    pd.to_datetime(df_tract_2['data_fi']) - pd.to_datetime(df_tract_2['start_window'])
    ).dt.days

df_tract_2.drop(columns=['cip_anterior', 'start_window', 'dbl'], inplace=True)

print(f'Number of treatments: {df_tract_2.shape[0]}')
df_tract_2.head()

Number of treatments: 451998


,id_cip_actual,data_inici,data_fi,atc,atc_desc
0,131,120,185,N02BF01,Gabapentina
1,252,-45,29,N02BF01,Gabapentina
2,252,34,390,N02BF01,Gabapentina
3,282,22,52,N02BF01,Gabapentina
4,463,-145,90,N02BF01,Gabapentina


### 3.2 Treatments Groupings
A dictionary of treatments agrupations by ATC (Anatomical Therapeutic Chemical) is defined to group the extracted treatments:

In [ ]:
atc_nivell_1 = {
    "A": "Fàrmacs Sistema digestiu i metabolisme",
    "B": "Fàrmacs Sang i òrgans hematopoètics",
    "C": "Fàrmacs Sistema cardiovascular",
    "D": "Fàrmacs Dermatològics",
    "G": "Fàrmacs Sistema genitourinari i hormones sexuals",
    "H": "Fàrmacs Preparats hormonals sistèmics",
    "J": "Fàrmacs Antiinfecciosos per a ús sistèmic",
    "L": "Fàrmacs Antineoplàsics i immunomoduladors",
    "M": "Fàrmacs Sistema musculoesquelètic",
    "N": "Fàrmacs Sistema nerviós",
    "P": "Fàrmacs Productes antiparasitaris, insecticides i repel·lents",
    "R": "Fàrmacs Sistema respiratori",
    "S": "Fàrmacs Òrgans dels sentits",
    "V": "Fàrmacs Diversos"
}

df_tract_2["atc_grup_codi"] = df_tract_2["atc"].str[0]                      # the group is defined by the 1st letter of the atc of the treatment
df_tract_2["atc_grup"] = df_tract_2["atc_grup_codi"].map(atc_nivell_1)      # mapping with groups
df_tract_2["atc_grup"] = df_tract_2["atc_grup"].fillna('Diversos')          # if no map, generic type assigned
df_tract_2.head()

,id_cip_actual,data_inici,data_fi,atc,atc_desc,atc_grup_codi,atc_grup
0,131,120,185,N02BF01,Gabapentina,N,Fàrmacs Sistema nerviós
1,252,-45,29,N02BF01,Gabapentina,N,Fàrmacs Sistema nerviós
2,252,34,390,N02BF01,Gabapentina,N,Fàrmacs Sistema nerviós
3,282,22,52,N02BF01,Gabapentina,N,Fàrmacs Sistema nerviós
4,463,-145,90,N02BF01,Gabapentina,N,Fàrmacs Sistema nerviós


>Transform data from long to patient-level format:

- Treatments are structured at the individual record level, where each row represents a single prescription assigned to a patient.
- To enable patient-level analysis, the data are transformed so that each patient is represented by a single row.

>Group treatments:

- Since patients may receive a large number of different treatments, prescriptions are aggregated by patient and pharmacological treatment group.
- The number of prescriptions within each treatment group is computed.
- NaN values are replaced with 0, indicating that no prescriptions were recorded for that group.

In [22]:
df_tract_grups = (
    df_tract_2.groupby(['id_cip_actual','atc_grup'], as_index=False)     # groups by cip and treatment grup keeping them as regular columns (not become indexes)
    .agg({'atc':'nunique'})                                              # number of different treatment codes per group
    .pivot(index='id_cip_actual', columns='atc_grup', values='atc')      # from long to wide format (cip as rows, atc_grup values as columns, counts of atc codes for grups as values)
    .reset_index()
    .fillna(0)                                                          # non-values as 0 counts for that diagnose group
)

# All columns that are not cip_anterior as integer type
cols = df_tract_grups.columns.difference(["id_cip_actual"])
df_tract_grups[cols] = df_tract_grups[cols].astype(int)

# Create a column of the total treatments for patient
df_tract_grups["Fàrmacs Totals"] = df_tract_grups[cols].sum(axis=1)
df_tract_grups.columns = df_tract_grups.columns.str.lower()
df_tract_grups.columns.name = None

In [23]:
df_tract_final = df_tract_grups.copy()
print(f'Number of treatments groups: {df_tract_final.shape[1]}')
print(f'Number patients with treatments: {df_tract_final.shape[0]}')
df_tract_final.to_csv('data/df_treatments.csv', index=False)
df_tract_final.head()

Number of treatments groups: 17
Number patients with treatments: 24976


,id_cip_actual,diversos,fàrmacs antiinfecciosos per a ús sistèmic,fàrmacs antineoplàsics i immunomoduladors,fàrmacs dermatològics,fàrmacs diversos,fàrmacs preparats hormonals sistèmics,"fàrmacs productes antiparasitaris, insecticides i repel·lents",fàrmacs sang i òrgans hematopoètics,fàrmacs sistema cardiovascular,fàrmacs sistema digestiu i metabolisme,fàrmacs sistema genitourinari i hormones sexuals,fàrmacs sistema musculoesquelètic,fàrmacs sistema nerviós,fàrmacs sistema respiratori,fàrmacs òrgans dels sentits,fàrmacs totals
0,2,0,0,0,0,0,0,0,0,1,0,1,0,0,0,2,4
1,7,0,0,0,0,0,0,0,0,3,2,0,1,1,0,0,7
2,8,0,0,0,0,0,0,0,1,3,4,1,0,0,0,4,13
3,9,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,3
4,12,0,1,0,1,0,0,0,0,1,0,1,0,2,0,0,6


## 4. Visits in Primary Care Centers

### 4.1 SQL Query

In [24]:
# Create chunks of patients as temporal SQL tables
chunk_size = 800
user_chunks_vap = create_chunks(cip_extraction, chunk_size, 'cip_anterior')

In [100]:
q_vap = """

 -- Insert the generated SQL as a Common Table Expression (CTE)
WITH patient_window AS (
    {usuaris}
)

SELECT 
    v.visi_usuari_cip AS cip_anterior,
    TO_CHAR(TO_DATE(TO_CHAR(v.visi_data_visita), 'J'), 'YYYY-MM-DD') AS data_visita,
    v.visi_up AS up,
    v.visi_tipus_visita AS tipus_visita,
    v.visi_servei_codi_servei AS codi_servei,
    v.visi_id AS visita_id

FROM vistb043{dbl} v

-- Join with the generated table
JOIN patient_window p
    ON v.visi_usuari_cip = p.cip
                
WHERE
    v.visi_up IN ('00041','00050','00051','00052','00060','00065','00066','00068','01097','04903','04904','04905','04906','04907')
    AND v.visi_situacio_visita = 'R'
    AND v.visi_servei_codi_servei IN ('INF','INFG','MG','EXTRA','TSOC','URGEN')
    AND v.visi_tipus_visita IN ('9C','CP','U','9E','9M','9T','TLF','9D','DA','DC','9R','PM')

    -- Only obtain the treatments in the temporal window of each patient
    AND TO_DATE(TO_CHAR(v.visi_data_visita), 'J') >= p.start_window
    AND TO_DATE(TO_CHAR(v.visi_data_visita), 'J') <  p.end_window
"""

In [101]:
df_vap = extract_data_parallel(
    q_vap,
    dblinks,
    user_chunks=user_chunks_vap,
    max_workers=6
)

print(f'Number of visits in primary care: {df_vap.shape[0]}')

Number of visits in primary care: 318354


Merge treatments with *cip_extraction* to get the *id_cip_actual* of each treatment:

In [ ]:
df_vap_2 = pd.merge(
    df_vap, 
    cip_extraction[['id_cip_actual', 'cip_anterior', 'start_window']],
    on='cip_anterior',
    how='left'
    )

df_vap_2['id_cip_actual'] = df_vap_2['id_cip_actual'].astype(int)
df_vap_2.insert(0, 'id_cip_actual', df_vap_2.pop('id_cip_actual'))

# Calculate the data taking start_window as day 1
df_vap_2['data'] = (
    pd.to_datetime(df_vap_2['data_visita']) - pd.to_datetime(df_vap_2['start_window'])
    ).dt.days
df_vap_2.drop(columns=['cip_anterior', 'data_visita', 'start_window', 'dbl'], inplace=True)
df_vap_2.insert(1, 'data', df_vap_2.pop('data'))

print(f'Number of primary care visits: {df_vap_2.shape[0]}')
df_vap_2.head()

Number of primary care visits: 318354


,id_cip_actual,data,up,tipus_visita,codi_servei,visita_id,dbl
0,8,122,00060,9E,MG,1155818400,@P6211_exp
1,8,190,00060,9T,MG,1156651743,@P6211_exp
2,8,193,00060,CP,EXTRA,1156680834,@P6211_exp
3,8,193,00060,9C,INF,1156726069,@P6211_exp
4,8,204,00060,9T,MG,1156726008,@P6211_exp


### 4.2 Aggregate Primary Care Visits

Create new columns *servei* and *tipus_visita* depending on the exctraction values:

In [103]:
df_vap_2['servei'] = np.where(
    df_vap_2['codi_servei'] == 'MG', 'metge',
    np.where(
    df_vap_2['codi_servei'].isin(['INF','INFG']), 'infermeria',
    np.where(
    df_vap_2['codi_servei'] == 'EXTRA', 'analitiques',
    np.where(
    df_vap_2['codi_servei'] == 'TSOC', 'treball_social',
        'urgencies_ap')
    )
    )
)

df_vap_2['tipus_visita_desc'] = np.where(
    df_vap_2['tipus_visita'].isin(['9C','CP','U']), 'presencial_reactiva',
    np.where(
    df_vap_2['tipus_visita'].isin(['9E','9M']), 'econsultes',
    np.where(
    df_vap_2['tipus_visita'].isin(['9T','TLF']), 'telefonica',
    np.where(
    df_vap_2['tipus_visita'].isin(['9D','DA','DC']), 'domicili',
        'presencial_proactiva')
    )
    )
)

>Transform data from long to patient-level format:

- Visit records are structured at the individual record level, where each row represents a single healthcare visit associated with a patient.
- To enable patient-level analysis, visits must be transformed so that each patient is represented by a single row.

>Aggregate healthcare visits:
- A new variable combining the service and visit type categories is created to facilitate grouping.
- Since patients may have a large number of healthcare visits, records are aggregated by patient and service–visit type combination
- The number of visits within each group is computed.
- NaN values are replaced with 0, indicating that no visits were recorded for that group.

In [ ]:
# Create a new column for grouping
df_vap_2['tipus_servei_visita'] = df_vap_2['servei'] + '_' + df_vap_2['tipus_visita_desc']

# Groupby and pivot
df_vap_g = (
    df_vap_2.groupby(['id_cip_actual', 'tipus_servei_visita'], as_index=False)           # groups by cip, service and type of visit keeping them as regular columns (not become indexes)
    .agg({'visita_id':'nunique'})                                                        # number of different visits for those groups
    .pivot(index='id_cip_actual', columns='tipus_servei_visita', values='visita_id')     # from long to wide format (cip as rows, tipus_visita values as columns, counts of visits for each type as values)
    .rename_axis(None, axis=1)
    .reset_index()
    .fillna(0)                                                                           # non-values as 0 counts for that diagnose group
)

# All columns that are not cip_anterior as integer type
cols = df_vap_g.columns.difference(["id_cip_actual"])
df_vap_g[cols] = df_vap_g[cols].astype(int)

# Create a column of the total visits for patient
df_vap_g.columns = df_vap_g.columns.str.lower()
df_vap_g.columns.name = None

# Groupby and calculate aggregations for data measures
df_vap_2['recent_vap'] = df_vap_2['data'] >= 275

df_vap_g_time = (
    df_vap_2.groupby('id_cip_actual')
    .agg(
        total_vap=('data', 'count'),     # number of total hospitalizations
        first_vap=('data', 'min'),       # date of 1st hospitalization
        last_vap=('data', 'max'),        # date of last hospitalization
        std_vap=('data', 'std'),         # std of hospitalizations
        recent_vap=('recent_vap', 'sum')
    )
).fillna(0).reset_index()

df_vap_g_time['std_vap'] = df_vap_g_time['std_vap'].round(1)
df_vap_g_time['days_since_last_vap'] = 365 - df_vap_g_time['last_vap']      # days since last hospitalization in relation to the end of the temporal window
df_vap_g_time.columns.name = None

In [105]:
# Merge all the aggregations
df_vap_final = (
    df_vap_g
    .merge(df_vap_g_time, on='id_cip_actual', how='left')
)

print(f'Number of different types of visits: {df_vap_final.shape[1]}')
print(f'Number patients with visits in primary care: {df_vap_final.shape[0]}')
df_vap_final.to_csv('data/df_ap_visits.csv', index=False)
df_vap_final.head()

Number of different types of visits: 30
Number patients with visits in primary care: 18273


,id_cip_actual,analitiques_domicili,analitiques_econsultes,analitiques_presencial_proactiva,analitiques_presencial_reactiva,infermeria_domicili,infermeria_econsultes,infermeria_presencial_proactiva,infermeria_presencial_reactiva,infermeria_telefonica,...,urgencies_ap_domicili,urgencies_ap_econsultes,urgencies_ap_presencial_reactiva,urgencies_ap_telefonica,total_vap,first_vap,last_vap,std_vap,recent_vap,days_since_last_vap
0,8,0,0,0,1,0,0,0,1,0,...,0,0,0,0,7,122,249,37.6,0,116
1,9,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,225,225,0.0,0,140
2,16,0,0,0,6,0,0,0,10,1,...,0,0,2,0,40,26,359,111.8,12,6
3,18,0,0,0,1,0,0,0,2,0,...,0,0,0,0,8,77,300,84.8,1,65
4,21,0,0,0,3,1,3,1,0,3,...,0,0,0,0,23,29,350,95.9,2,15


## 5. Extraction of Hospitalizations
- In the case of Intermediate Hospital Care Visits, data doesn't come from ECAP and it cannot be directly extracted from the SQL database, instead data comes from SAPBO and it is charged in .csv format.

### 5.1 Hospitalization of Intermediate Care

#### 5.1.1 Load Data

In [106]:
# Load data
df_hosp = (
    pd.read_csv('data/cmbd_hosp.csv')
    [['CIP (14d)',
      'UP codi',
      "Data d'inici d'assistència (AAAAMMDD)",
      "Data final d'assistència (AAAAMMDD)",
      "Circumstàncies d'admissió desc",
      "Tipus d'activitat desc",
      "Procedència desc"]]
    .rename({"CIP (14d)":'cip_anterior_14',
             "UP codi":'up',
             "Data d'inici d'assistència (AAAAMMDD)":'data_ingres',
             "Data final d'assistència (AAAAMMDD)":'data_alta',
             "Circumstàncies d'admissió desc":'tipus_ingres',
             "Tipus d'activitat desc":'tipus_activitat',
             "Procedència desc":'procedencia'}, axis=1)
)

print(f'Number of Intermidiate Hospitalizations: {df_hosp.shape[0]}')

Number of Intermidiate Hospitalizations: 82415


Merge intermediate hospitalizations with *cip_extraction* to get the *id_cip_actual* and temporal window of each one:

In [107]:
df_hosp_2 = pd.merge(
    df_hosp,
    cip_extraction[['id_cip_actual', 'cip_anterior_14', 'start_window', 'end_window']],
    on='cip_anterior_14',
    how='left'
).dropna().reset_index(drop=True)

df_hosp_2['id_cip_actual'] = df_hosp_2['id_cip_actual'].astype(int)
df_hosp_2.insert(0, 'id_cip_actual', df_hosp_2.pop('id_cip_actual'))

# Calculate the data taking start_window as day 1
df_hosp_2['data'] = (
    pd.to_datetime(df_hosp_2['data_ingres']) - pd.to_datetime(df_hosp_2['start_window'])
    ).dt.days

# Filter all emergencies that are inside the temporal window of each patient
df_hosp_2 = df_hosp_2[
    (pd.to_datetime(df_hosp_2['start_window']) <= pd.to_datetime(df_hosp_2['data_ingres'])) & 
    (pd.to_datetime(df_hosp_2['data_ingres']) < pd.to_datetime(df_hosp_2['end_window']))
    ].reset_index(drop=True)

df_hosp_2.drop(columns=['cip_anterior_14', 'start_window', 'end_window'], inplace=True)
df_hosp_2.insert(1, 'data', df_hosp_2.pop('data'))

print(f'Number of Intermidiate Hospitalizations: {df_hosp_2.shape[0]}')

Number of Intermidiate Hospitalizations: 8846


New calculated columns are added to capture relevant information:

In [108]:
# How many days the patient has been hospitalized
df_hosp_2['dies_estada'] = (
    (pd.to_datetime(df_hosp_2['data_alta']) - pd.to_datetime(df_hosp_2['data_ingres']))
    .dt.days
)

# Hospitalization days Agrupation
df_hosp_2['grup_dies_estada'] = np.where(
    df_hosp_2['dies_estada']<2, '<02',
    np.where(
    df_hosp_2['dies_estada']<7, '<07',
    np.where(
    df_hosp_2['dies_estada']<14, '<14', '>14'
)))

# Patient Origin Agrupation: from 5 classifications to 2 (domicili_o_residencia, trasllat)
df_hosp_2['grup_procedencia'] = np.where(
    df_hosp_2['procedencia'].str.contains('domicili', na=False, case=False),
    'domicili_o_residencia',
    'trasllat'
)

- Select the rows that correspond to the relevant *UP* (Unitat Productiva).
- Drop irrelevant columns that were used to do some calculations but are not useful anymore.

In [109]:
df_hosp_2 = df_hosp_2[df_hosp_2['up'].isin([39, 763, 826, 86, 737])].reset_index(drop=True)
df_hosp_2 = df_hosp_2[['id_cip_actual',
                   'data', 
                   'grup_dies_estada',
                   'grup_procedencia',
                   'tipus_ingres',
                   'tipus_activitat']].reset_index(drop=True)

df_hosp_2.head()

,id_cip_actual,data,grup_dies_estada,grup_procedencia,tipus_ingres,tipus_activitat
0,21716,39,<02,domicili_o_residencia,Admissió urgent,Hospitalització convencional
1,67641,17,<07,domicili_o_residencia,Admissió urgent,Hospitalització convencional
2,18336,127,<02,domicili_o_residencia,Admissió urgent,Hospitalització convencional
3,99477,112,<02,domicili_o_residencia,Admissió urgent,Hospitalització convencional
4,109568,297,<02,domicili_o_residencia,Admissió programada,Cirurgia major ambulatòria (CMA)


#### 5.1.2 Hospitalization of Intermediate Care Agrupations

>Transform data from long to patient-level format:
- Hospitalization records are structured at the individual record level, where each row represents a single hospitalization episode associated with a patient.
- To enable patient-level analysis, the data are transformed so that each patient is represented by a single row

>Aggregate hospitalization variables:
- Classification labels are simplified to shorter and more interpretable names.
- A new variable combining *grup_procedencia*, *tipus_ingres*, and *tipus_activitat* is created to facilitate grouping.
- Hospitalization records are aggregated at the patient level and several summary variables are computed:
    1) Counts for all combinations of *grup_procedencia*, *tipus_ingres* and *tipus_activitat*
    2) Counts for each *grup_dies_estada* category.
    3) Temporal features derived from data, including the total number of hospitalizations, the first and last hospitalization dates, and the standard deviation between hospitalizations.
    4) The number of hospitalizations occurring within the last 90 days of the observation window.

In [110]:
# Create shorter name for classifications for better pivoting
map_grup = {
    'domicili_o_residencia' : 'DR', 
    'trasllat' : 'T'
    }

df_hosp_2['grup_procedencia'] = df_hosp_2['grup_procedencia'].map(map_grup)

map_ingres = {
    'Admissió urgent' : 'AU', 
    'Admissió programada' : 'AP'
    }

df_hosp_2['tipus_ingres'] = df_hosp_2['tipus_ingres'].map(map_ingres)

map_activitat = {
    'Hospitalització convencional' : 'HC', 
    'Cirurgia major ambulatòria (CMA)' : 'CMA',
    'Hospitalització domiciliària' : 'HD'}

df_hosp_2['tipus_activitat'] = df_hosp_2['tipus_activitat'].map(map_activitat)

# Define new columns of Argupations
df_hosp_2['proc_ingres_activitat'] = df_hosp_2['grup_procedencia'] + '_' + df_hosp_2['tipus_ingres'] + '_' + df_hosp_2['tipus_activitat']
df_hosp_2['grup_dies_estada'] = 'hospitalitzacio_' + df_hosp_2['grup_dies_estada']

In [111]:
# Group and pivot by procediment, ingrés, activitat: counts for each possible combination
df_hosp_g = (
    df_hosp_2.pivot_table(         
        index='id_cip_actual',
        columns='proc_ingres_activitat',
        aggfunc='size',
        fill_value=0
    )
    ).reset_index()

df_hosp_g.columns.name = None

# Group and pivot by dies estada: counts for each type of estada
df_hosp_g_dies = (
    df_hosp_2.pivot_table(        
        index='id_cip_actual',
        columns='grup_dies_estada',
        aggfunc='size',
        fill_value=0
        )
    ).reset_index()

df_hosp_g_dies.columns.name = None

# Groupby and calculate aggregations for data measures
df_hosp_2['recent_hosp'] = df_hosp_2['data'] >= 275

df_hosp_g_time = (
    df_hosp_2.groupby('id_cip_actual')
    .agg(
        total_hosp=('data', 'count'),     # number of total hospitalizations
        first_hosp=('data', 'min'),       # date of 1st hospitalization
        last_hosp=('data', 'max'),        # date of last hospitalization
        std_hosp=('data', 'std'),         # std of hospitalizations
        recent_hosp=('recent_hosp', 'sum')
    )
).fillna(0).reset_index()

df_hosp_g_time['std_hosp'] = df_hosp_g_time['std_hosp'].round(1)
df_hosp_g_time['days_since_last_hosp'] = 365 - df_hosp_g_time['last_hosp']      # days since last hospitalization in relation to the end of the temporal window
df_hosp_g_time.columns.name = None

In [127]:
# Merge all the aggregations
df_hosp_final = (
    df_hosp_g
    .merge(df_hosp_g_dies, on='id_cip_actual', how='left')
    .merge(df_hosp_g_time, on='id_cip_actual', how='left')
)

df_hosp_final.columns = df_hosp_final.columns.str.lower()

df_hosp_final.rename(columns={
    'hospitalitzacio_<02': 'hospitalitzacio_less_2_days',
    'hospitalitzacio_<07' : 'hospitalitzacio_less_7_days',
    'hospitalitzacio_<14' : 'hospitalitzacio_less_14_days',
    'hospitalitzacio_>14' : 'hospitalitzacio_more_14_days'
}, inplace=True)

print(f'Number of different types of hospitalizations: {df_hosp_final.shape[1]}')
print(f'Number patients with hospitalizations: {df_vap_final.shape[0]}')
df_hosp_final.to_csv('data/df_hosp.csv')
df_hosp_final.head()

Number of different types of hospitalizations: 21
Number patients with hospitalizations: 18273


,id_cip_actual,dr_ap_cma,dr_ap_hc,dr_ap_hd,dr_au_cma,dr_au_hc,dr_au_hd,t_ap_cma,t_ap_hc,t_ap_hd,...,hospitalitzacio_less_2_days,hospitalitzacio_less_7_days,hospitalitzacio_less_14_days,hospitalitzacio_more_14_days,total_hosp,first_hosp,last_hosp,std_hosp,recent_hosp,days_since_last_hosp
0,8,0,0,0,0,1,0,0,0,0,...,1,0,0,0,1,318,318,0.0,1,47
1,12,1,0,0,0,0,0,0,0,0,...,1,0,0,0,1,127,127,0.0,0,238
2,16,0,0,0,0,1,0,0,0,0,...,1,0,0,0,1,100,100,0.0,0,265
3,21,0,0,0,0,2,0,0,0,0,...,2,0,0,0,2,296,302,4.2,2,63
4,52,1,0,0,0,0,0,0,0,0,...,1,0,0,0,1,234,234,0.0,0,131


### 5.2 Emergencies

#### 5.2.1 Load data

In [113]:
# Load data
df_urg = (
    pd.read_csv('data/cmbd_urg.csv')
    [['CIP (14d)',
      "Data final d'assistència (DDMMAAAA)",
      "Nivell triatge final desc",
      "Iniciativa de la utilització d'urgències desc"]]
    .rename({"CIP (14d)":'cip_anterior_14',
             "Data final d'assistència (DDMMAAAA)":'data_urgencia',
             "Nivell triatge final desc":'nivell_triatge',
             "Iniciativa de la utilització d'urgències desc":'iniciativa'}, axis=1)
)
print(f'Number of Emergencies: {df_urg.shape[0]}')

/tmp/ipykernel_55378/1964115136.py:3: DtypeWarning: Columns (0: Iniciativa de la utilització d'urgències codi, 1: Mitjà d'arribada a urgències) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('data/cmbd_urg.csv')


Number of Emergencies: 324745


Merge emergencies with *cip_extraction* to get the *id_cip_actual* and temporal window of each one:

In [114]:
df_urg_2 = pd.merge(
    df_urg,
    cip_extraction[['id_cip_actual', 'cip_anterior_14','start_window','end_window']],
    on='cip_anterior_14',
    how='left'
).dropna().reset_index(drop=True)

df_urg_2['id_cip_actual'] = df_urg_2['id_cip_actual'].astype(int)
df_urg_2.insert(0, 'id_cip_actual', df_urg_2.pop('id_cip_actual'))

# Filter all emergencies that are inside the temporal window of each patient
df_urg_2 = df_urg_2[
    (pd.to_datetime(df_urg_2['start_window']) <= pd.to_datetime(df_urg_2['data_urgencia'])) & 
    (pd.to_datetime(df_urg_2['data_urgencia']) < pd.to_datetime(df_urg_2['end_window']))
    ].reset_index(drop=True)

# Calculate the data taking start_window as day 1
df_urg_2['data'] = (
    np.ceil(
        (pd.to_datetime(df_urg_2['data_urgencia']) - pd.to_datetime(df_urg_2['start_window'])).dt.days
    ).astype(int)
)

# Remove unuseful columns
df_urg_2.drop({'data_urgencia', 'start_window', 'end_window', 'cip_anterior_14'}, axis=1, inplace=True)

print(f'Number of Emergencies: {df_urg_2.shape[0]}')
df_urg_2.head()

Number of Emergencies: 14060


,id_cip_actual,nivell_triatge,iniciativa,data
0,26965,Risc vital potencial,Pròpia,39
1,63954,Risc vital potencial,Pròpia,66
2,110195,Risc vital potencial,Pròpia,26
3,45751,Risc vital potencial,Pròpia,30
4,49230,Risc vital potencial,Pròpia,7


New calculated columns are added to capture relevant information:

In [115]:
# New category in iniciativa for agrupation: altres
df_urg_2['iniciativa'] = np.where(
    df_urg_2['iniciativa'].str.contains('N/D|judicial'), 'altres', df_urg_2['iniciativa']
)

#### 5.2.2 Emergency Agrupations

>Transform data from long to patient-level format:
- Emergency visit records are structured at the individual record level, where each row represents a single emergency department visit associated with a patient.
- o enable patient-level analysis, the data are transformed so that each patient is represented by a single row.

>Aggregate emergency visit variables:
- A new variable combining *nivell_triatge* and *iniciativa* is created to facilitate grouping.
- Emergency visit records are aggregated at the patient level and several summary variables are computed:
    1) Counts for all combinations of *nivell_triatge* and *iniciativa*
    2) Temporal features derived from data, including the total number of emergency visits, the first and last emergency visit dates, and the standard deviation between visits.
    4) The number of emergency visits occurring within the last 90 days of the observation window.

In [116]:
# Create a new column for grouping
df_urg_2['triatge_iniciativa'] = df_urg_2['nivell_triatge'] + '_' + df_urg_2['iniciativa']

In [117]:
# Group and pivot by triatge and iniciativa
df_urg_g = (
    df_urg_2.pivot_table(
        index='id_cip_actual',
        columns='triatge_iniciativa',
        aggfunc='size',
        fill_value=0
    )
    ).reset_index()

df_urg_g.columns.name = None

# Groupby and calculate aggregations for data measures
df_urg_2['recent_urg'] = df_urg_2['data'] >= 275    # mask of emergency wihtin the last 90 days of the temporal window

df_urg_g_time = (
    df_urg_2.groupby('id_cip_actual')
    .agg(
        total_urg=('data', 'count'),      # number of total emergencies
        first_urg=('data', 'min'),        # date of 1st emergencies
        last_urg=('data', 'max'),         # date of last emergencies
        std_urg=('data', 'std'),           # std of emergencies
        recent_urg=('recent_urg', 'sum')
        )
    ).fillna(0).reset_index()

df_urg_g_time['std_urg'] = df_urg_g_time['std_urg'].round(1)
df_urg_g_time['days_since_last_urg'] = 365 - df_urg_g_time['last_urg']      # days since last emergency in relation to the end of the temporal window
df_hosp_g_time.columns.name = None

In [118]:
# Merge all the aggregations
df_urg_final = (
    df_urg_g
    .merge(df_urg_g_time, on='id_cip_actual', how='left')
)

df_urg_final.columns = df_urg_final.columns.str.lower()
df_urg_final.columns = df_urg_final.columns.str.replace(' ', '_')
print(f'Number of different types of emergencies: {df_urg_final.shape[1]}')
print(f'Number patients with emergencies: {df_urg_final.shape[0]}')
df_urg_final.to_csv('data/df_urg.csv')
df_urg_final.head()

Number of different types of emergencies: 21
Number patients with emergencies: 6846


,id_cip_actual,n/d_professional_sanitari,n/d_pròpia,n/d_altres,no_urgent_professional_sanitari,no_urgent_pròpia,risc_vital_immediat_professional_sanitari,risc_vital_immediat_pròpia,risc_vital_potencial_professional_sanitari,risc_vital_potencial_pròpia,...,risc_vital_previsible_pròpia,risc_vital_previsible_altres,sense_risc_vital_professional_sanitari,sense_risc_vital_pròpia,total_urg,first_urg,last_urg,std_urg,recent_urg,days_since_last_urg
0,8,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,318,318,0.0,1,47
1,16,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,101,101,0.0,0,264
2,21,0,0,0,0,0,0,0,1,1,...,0,0,0,0,3,33,303,154.2,2,62
3,52,0,0,0,0,0,0,0,0,1,...,0,0,0,1,2,100,240,99.0,0,125
4,69,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,298,298,0.0,1,67


## 6. Final Dataset

At this stage, all healthcare information is aggregated and merged at the patient level. The resulting dataset combines demographic, clinical, pharmacological, laboratory, and healthcare utilization variables and serves as the input for the predictive modeling experiments.

### 6.1 Lab Tests

In [132]:
df_lab_merge = pd.read_csv('data/df_lab.csv')

df_final = pd.merge(
    df_usuaris,
    df_lab_merge,
    on='id_cip_actual',
    how='left'
).fillna(0)

print(f'Number of users: {df_final.shape[0]}')
print(f'Shape: {df_final.shape}')
df_final.head()

Number of users: 55979
Shape: (55979, 66)


,id_cip_actual,sexe,situacio,cronic,gma_plng,gma_pniv,grup_edat_75-80,grup_edat_80-85,grup_edat_85-90,grup_edat_major_90,...,last_ferritina-sèrum,last_ferro-sèrum,last_fosfatasa alcalina-sèrum,last_glucosa-sèrum,last_pro-bnp-sèrum,last_proteïna c reactiva (pcr)-sèrum,last_proteïna-sèrum,last_tirotropina-sèrum,last_urea-sèrum,last_àcid fòlic-sèrum
0,2,0,0,NO,0.021,2.0,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3,1,0,NO,0.000,0.0,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5,1,0,NO,0.000,0.0,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7,0,0,NO,0.021,2.0,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8,0,1,NO,0.000,0.0,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 6.2 Diagnoses

In [133]:
df_dx_merge = pd.read_csv('data/df_diagnoses.csv')
cols = df_dx_merge.columns.difference(['id_cip_actual'])

df_final_2 = pd.merge(
    df_final,
    df_dx_merge,
    on='id_cip_actual',
    how='left'
).fillna(0)

df_final_2[cols] = df_final_2[cols].astype(int)

print(f'Number of users: {df_final_2.shape[0]}')
print(f'Shape: {df_final_2.shape}')
df_final_2.head()

Number of users: 55979
Shape: (55979, 75)


,id_cip_actual,sexe,situacio,cronic,gma_plng,gma_pniv,grup_edat_75-80,grup_edat_80-85,grup_edat_85-90,grup_edat_major_90,...,last_àcid fòlic-sèrum,problemes_salut_aguts,problemes_salut_anomalia_congenita,problemes_salut_cronics,problemes_salut_indefinits,problemes_salut_neoplasia_benigna,problemes_salut_neoplasia_maligna,signes_i_sintomes,altres_diags,diags_totals
0,2,0,0,NO,0.021,2.0,1,0,0,0,...,0.0,0,0,0,0,0,0,0,3,3
1,3,1,0,NO,0.000,0.0,1,0,0,0,...,0.0,0,0,0,0,0,0,0,0,0
2,5,1,0,NO,0.000,0.0,1,0,0,0,...,0.0,0,0,0,0,0,0,0,0,0
3,7,0,0,NO,0.021,2.0,1,0,0,0,...,0.0,0,0,0,0,0,0,0,7,7
4,8,0,1,NO,0.000,0.0,1,0,0,0,...,0.0,0,0,0,0,0,0,0,4,4


### 6.3 Treatments

In [134]:
df_tract_merge = pd.read_csv('data/df_treatments.csv')
cols = df_tract_merge.columns.difference(['id_cip_actual'])

df_final_3 = pd.merge(
    df_final_2,
    df_tract_merge,
    on='id_cip_actual',
    how='left'
).fillna(0)

df_final_3[cols] = df_final_3[cols].astype(int)

print(f'Number of users: {df_final_3.shape[0]}')
print(f'Shape: {df_final_3.shape}')
df_final_3.head()

Number of users: 55979
Shape: (55979, 91)


,id_cip_actual,sexe,situacio,cronic,gma_plng,gma_pniv,grup_edat_75-80,grup_edat_80-85,grup_edat_85-90,grup_edat_major_90,...,"fàrmacs productes antiparasitaris, insecticides i repel·lents",fàrmacs sang i òrgans hematopoètics,fàrmacs sistema cardiovascular,fàrmacs sistema digestiu i metabolisme,fàrmacs sistema genitourinari i hormones sexuals,fàrmacs sistema musculoesquelètic,fàrmacs sistema nerviós,fàrmacs sistema respiratori,fàrmacs òrgans dels sentits,fàrmacs totals
0,2,0,0,NO,0.021,2.0,1,0,0,0,...,0,0,1,0,1,0,0,0,2,4
1,3,1,0,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,1,0,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,7,0,0,NO,0.021,2.0,1,0,0,0,...,0,0,3,2,0,1,1,0,0,7
4,8,0,1,NO,0.000,0.0,1,0,0,0,...,0,1,3,4,1,0,0,0,4,13


### 6.4 Primary Care Visits

In [135]:
df_vap_merge = pd.read_csv('data/df_ap_visits.csv')
cols = df_vap_merge.columns.difference(['id_cip_actual'])

df_final_4 = pd.merge(
    df_final_3,
    df_vap_merge,
    on='id_cip_actual',
    how='left'
).fillna(0)

df_final_4[cols] = df_final_4[cols].astype(int)

print(f'Number of users: {df_final_4.shape[0]}')
print(f'Shape: {df_final_4.shape}')
df_final_4.head()

Number of users: 55979
Shape: (55979, 120)


,id_cip_actual,sexe,situacio,cronic,gma_plng,gma_pniv,grup_edat_75-80,grup_edat_80-85,grup_edat_85-90,grup_edat_major_90,...,urgencies_ap_domicili,urgencies_ap_econsultes,urgencies_ap_presencial_reactiva,urgencies_ap_telefonica,total_vap,first_vap,last_vap,std_vap,recent_vap,days_since_last_vap
0,2,0,0,NO,0.021,2.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3,1,0,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,1,0,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,7,0,0,NO,0.021,2.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,1,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,7,122,249,37,0,116


### 6.5 Hospitalizations

In [136]:
df_hosp_merge = pd.read_csv('data/df_hosp.csv')
cols = df_hosp_merge.columns.difference(['id_cip_actual'])

df_final_5 = pd.merge(
    df_final_4,
    df_hosp_merge,
    on='id_cip_actual',
    how='left'
).fillna(0)

df_final_5[cols] = df_final_5[cols].astype(int)

print(f'Number of users: {df_final_5.shape[0]}')
print(f'Shape: {df_final_5.shape}')
df_final_5.head()

Number of users: 55979
Shape: (55979, 141)


,id_cip_actual,sexe,situacio,cronic,gma_plng,gma_pniv,grup_edat_75-80,grup_edat_80-85,grup_edat_85-90,grup_edat_major_90,...,hospitalitzacio_less_2_days,hospitalitzacio_less_7_days,hospitalitzacio_less_14_days,hospitalitzacio_more_14_days,total_hosp,first_hosp,last_hosp,std_hosp,recent_hosp,days_since_last_hosp
0,2,0,0,NO,0.021,2.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3,1,0,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,1,0,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,7,0,0,NO,0.021,2.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,1,NO,0.000,0.0,1,0,0,0,...,1,0,0,0,1,318,318,0,1,47


### 6.6 Emergencies

In [137]:
df_urg_merge = pd.read_csv('data/df_urg.csv')
df_urg_merge.drop(columns=['Unnamed: 0'], inplace=True)
cols = df_urg_merge.columns.difference(['id_cip_actual'])

df_final_6 = pd.merge(
    df_final_5,
    df_urg_merge,
    on='id_cip_actual',
    how='left'
).fillna(0)

df_final_6[cols] = df_final_6[cols].astype(int)

print(f'Number of users: {df_final_6.shape[0]}')
print(f'Shape: {df_final_6.shape}')
df_final_6.head()

Number of users: 55979
Shape: (55979, 161)


,id_cip_actual,sexe,situacio,cronic,gma_plng,gma_pniv,grup_edat_75-80,grup_edat_80-85,grup_edat_85-90,grup_edat_major_90,...,risc_vital_previsible_pròpia,risc_vital_previsible_altres,sense_risc_vital_professional_sanitari,sense_risc_vital_pròpia,total_urg,first_urg,last_urg,std_urg,recent_urg,days_since_last_urg
0,2,0,0,NO,0.021,2.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3,1,0,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,1,0,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,7,0,0,NO,0.021,2.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,1,NO,0.000,0.0,1,0,0,0,...,0,0,0,0,1,318,318,0,1,47


### 6.7 Export final Dataset

In [138]:
df_final_6.to_csv('data/dataset.csv', index=False)